# Medical Prediction Model Evaluation — End-to-End Walkthrough

**Based on:** Van Calster et al. (2025) *"Evaluation of performance measures in predictive AI models to support medical decisions: overview and guidance."* The Lancet Digital Health.

This notebook demonstrates the complete evaluation framework using the ADNEX model case study (894 patients with ovarian tumours). We cover:

1. **Data Loading & Exploration**
2. **Discrimination** — AUROC, AUPRC, pAUROC
3. **Calibration** — O:E ratio, intercept, slope, calibration plot
4. **Overall Performance** — Brier score, logloss, R² measures
5. **Classification** — Accuracy, F1, MCC, sensitivity, specificity
6. **Clinical Utility** — Net benefit, decision curve analysis
7. **Logistic Recalibration** — Before vs. after comparison
8. **Bootstrap Confidence Intervals**
9. **Full Recommended Evaluation** — The paper's core set of measures and plots

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

---
## 1. Data Loading & Exploration

The ADNEX model predicts the probability of malignancy in women with an ovarian tumour. The TransIOTA external validation dataset has **894 patients** from 6 centres in 4 countries. Prevalence of malignancy is **49%**.

In [ ]:
from medpred.utils.data import load_case_study_data

y_true, y_prob = load_case_study_data()

print(f"Patients: {len(y_true)}")
print(f"Malignant (event=1): {np.sum(y_true)} ({100*np.mean(y_true):.1f}%)")
print(f"Benign (event=0):    {np.sum(y_true==0)} ({100*np.mean(y_true==0):.1f}%)")
print(f"\nPredicted probability range: [{y_prob.min():.4f}, {y_prob.max():.4f}]")
print(f"Mean predicted probability:  {y_prob.mean():.4f}")
print(f"Observed prevalence:         {y_true.mean():.4f}")

In [ ]:
# Quick look at the data distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(y_prob[y_true==0], bins=30, alpha=0.7, color='#93c5fd', label='Benign', edgecolor='white')
axes[0].hist(y_prob[y_true==1], bins=30, alpha=0.7, color='#fca5a5', label='Malignant', edgecolor='white')
axes[0].set_xlabel('Predicted Probability of Malignancy')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Predictions')
axes[0].legend()

axes[1].boxplot([y_prob[y_true==0], y_prob[y_true==1]], labels=['Benign', 'Malignant'])
axes[1].set_ylabel('Predicted Probability')
axes[1].set_title('Predictions by Outcome')

plt.tight_layout()
plt.show()

---
## 2. Discrimination

Discrimination measures how well the model separates events from non-events based on **ranks** of predicted probabilities.

| Measure | Properness | Focus | Recommendation |
|---------|-----------|-------|----------------|
| **AUROC** | Semi-proper | Clear | **Recommended** |
| AUPRC | Semi-proper | Unclear | Inadvisable |
| pAUROC | Semi-proper | Unclear | Inadvisable |

In [ ]:
from medpred.metrics.discrimination import discrimination_metrics, roc_curve_data, pr_curve_data

disc = discrimination_metrics(y_true, y_prob, min_sensitivity=0.8)

print("Discrimination Measures")
print("=" * 40)
print(f"  AUROC (C-statistic):  {disc['auroc']:.3f}  ← RECOMMENDED")
print(f"  AUPRC (Avg Precision): {disc['auprc']:.3f}  (inadvisable)")
print(f"  pAUROC (sens ≥ 0.8):  {disc['pauroc']:.3f}  (inadvisable)")

In [ ]:
from medpred.visualization import plot_roc_curve, plot_pr_curve

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
plot_roc_curve(y_true, y_prob, auroc=disc['auroc'], ax=ax1)
plot_pr_curve(y_true, y_prob, auprc=disc['auprc'], ax=ax2)
plt.tight_layout()
plt.show()

**Key insight:** AUROC = 0.911 indicates excellent discrimination. The model assigns higher probabilities to malignant tumours than benign ones. However, good discrimination alone does NOT mean the model is clinically useful — we must also check calibration and clinical utility.

---
## 3. Calibration

Calibration assesses whether predicted probabilities correspond to observed event proportions. Three levels:
- **Mean calibration:** O:E ratio, calibration intercept
- **Weak calibration:** calibration slope
- **Moderate calibration:** calibration plot (most informative)

All calibration measures are **semi-proper** with **clear focus** on statistical performance.

In [ ]:
from medpred.metrics.calibration import calibration_metrics

cal = calibration_metrics(y_true, y_prob)

print("Calibration Measures")
print("=" * 50)
print(f"  O:E ratio:              {cal['oe_ratio']:.3f}  (ideal = 1.0)")
print(f"  Calibration intercept:  {cal['calibration_intercept']:.3f}  (ideal = 0.0)")
print(f"  Calibration slope:      {cal['calibration_slope']:.3f}  (ideal = 1.0)")
print(f"  ECI:                    {cal['eci']:.3f}  (ideal = 0.0)")
print(f"  ICI:                    {cal['ici']:.3f}  (ideal = 0.0)")
print(f"  ECE:                    {cal['ece']:.3f}  (ideal = 0.0)")

print(f"\n  → O:E = {cal['oe_ratio']:.3f} means {100*(cal['oe_ratio']-1):.0f}% more events observed than expected")
print(f"  → Intercept > 0 means the model underestimates risk on average")
print(f"  → Slope ≈ 1 means the spread of probabilities is adequate")

In [ ]:
from medpred.visualization import plot_calibration

fig, ax = plt.subplots(figsize=(7, 7))
plot_calibration(y_true, y_prob, ax=ax)
ax.set_title('Calibration Plot — ADNEX Model (RECOMMENDED)', fontweight='bold')
plt.show()

print("\nThe curve lies mostly ABOVE the diagonal → the model underestimates")
print("the probability of malignancy. Likely because 5 of 6 validation centres")
print("were tertiary care, resulting in a higher prevalence (49%) than expected.")

---
## 4. Overall Performance

Overall performance combines discrimination and calibration. These measures are more relevant for model selection tasks than for clinical validation.

| Measure | Properness | Notes |
|---------|-----------|-------|
| Loglikelihood, Logloss, Brier | **Strictly proper** | Gold standard |
| Scaled Brier, R² variants | Asympt. strictly proper | Relative to null model |
| Discrimination slope, MAPE | Improper | Inadvisable |

In [ ]:
from medpred.metrics.overall import overall_metrics

ov = overall_metrics(y_true, y_prob)

print("Overall Performance Measures")
print("=" * 55)
print("\n  Strictly proper (trustworthy):")
print(f"    Loglikelihood:     {ov['loglikelihood']:.1f}")
print(f"    Logloss:           {ov['logloss']:.1f}")
print(f"    Brier score:       {ov['brier_score']:.3f}  (0 = perfect, 1 = worst)")

print("\n  Asymptotically strictly proper:")
print(f"    Scaled Brier/IPA:  {ov['scaled_brier']:.3f}  (0 = null model, 1 = perfect)")
print(f"    McFadden R²:       {ov['mcfadden_r2']:.3f}")
print(f"    Cox-Snell R²:      {ov['cox_snell_r2']:.3f}")
print(f"    Nagelkerke R²:     {ov['nagelkerke_r2']:.3f}")

print("\n  Improper (inadvisable):")
print(f"    Disc. slope:       {ov['discrimination_slope']:.3f}")
print(f"    MAPE:              {ov['mape']:.3f}")

---
## 5. Classification (at threshold t = 0.1)

A threshold of 10% is commonly recommended for ADNEX. This implies accepting up to **9 false positives per true positive** — i.e., the benefit of detecting a malignancy is considered 9× greater than the harm of unnecessary advanced surgery.

**Important:** ALL classification summary measures are **improper** at clinically relevant thresholds (not t=0.5 or t=prevalence). The paper recommends against using them for clinical validation.

In [ ]:
from medpred.metrics.classification import classification_metrics

cl = classification_metrics(y_true, y_prob, threshold=0.1)

print(f"Classification at threshold t = {cl['threshold']}")
print("=" * 55)
print(f"\n  Confusion Matrix:")
print(f"                    Predicted High  Predicted Low")
print(f"    Malignant:       TP = {cl['tp']:>4d}       FN = {cl['fn']:>4d}")
print(f"    Benign:          FP = {cl['fp']:>4d}       TN = {cl['tn']:>4d}")

print(f"\n  Summary measures (ALL IMPROPER at t=0.1):")
print(f"    Accuracy:           {cl['accuracy']:.3f}")
print(f"    Balanced accuracy:  {cl['balanced_accuracy']:.3f}")
print(f"    Youden index:       {cl['youden_index']:.3f}")
print(f"    DOR:                {cl['diagnostic_odds_ratio']:.1f}")
print(f"    Kappa:              {cl['kappa']:.3f}")
print(f"    F1 score:           {cl['f1_score']:.3f}  ⚠ IMPROPER + UNCLEAR FOCUS")
print(f"    MCC:                {cl['mcc']:.3f}")

print(f"\n  Descriptive partial measures (report in pairs):")
print(f"    Sensitivity:        {cl['sensitivity']:.3f}  (with specificity)")
print(f"    Specificity:        {cl['specificity']:.3f}  (with sensitivity)")
print(f"    PPV (precision):    {cl['ppv']:.3f}  (with NPV)")
print(f"    NPV:                {cl['npv']:.3f}  (with PPV)")

In [ ]:
from medpred.visualization import plot_classification_at_thresholds

fig, ax = plt.subplots(figsize=(9, 5))
plot_classification_at_thresholds(y_true, y_prob, ax=ax)
ax.axvline(x=0.1, color='black', linestyle='--', alpha=0.5, label='t=0.1')
ax.legend()
plt.show()

---
## 6. Clinical Utility — The Most Important Domain

Clinical utility explicitly incorporates misclassification costs following decision-analytical principles. This is what determines whether a model should actually be **used in practice**.

**Key question:** Does the model lead to better decisions than the reference strategies (treat all / treat none)?

In [ ]:
from medpred.metrics.clinical_utility import clinical_utility_metrics

cu = clinical_utility_metrics(y_true, y_prob, threshold=0.1, cost_fn_ratio=0.9)

print("Clinical Utility Measures")
print("=" * 55)
print(f"  Decision threshold:        t = {0.1}")
print(f"  Implied cost ratio:        FN is {int(0.9/0.1)}× worse than FP")
print(f"")
print(f"  Net benefit (model):       {cu['net_benefit']:.3f}")
print(f"  Net benefit (treat all):   {cu['treat_all_nb']:.3f}")
print(f"  Net benefit (treat none):  {cu['treat_none_nb']:.3f}")
print(f"")
print(f"  → Model NB ({cu['net_benefit']:.3f}) > Treat All ({cu['treat_all_nb']:.3f}) > Treat None (0)")
print(f"  → The model ADDS clinical value at this threshold!")
print(f"")
print(f"  Standardized NB:           {cu['standardized_net_benefit']:.3f}  (max = 1.0)")
print(f"  Expected cost:             {cu['expected_cost']:.3f}")
print(f"  EC optimal threshold:      {cu['expected_cost_threshold']:.3f}")

In [ ]:
from medpred.visualization import plot_decision_curve, plot_expected_cost_curve

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

plot_decision_curve(y_true, y_prob, threshold_range=(0.01, 0.99), ax=ax1)
ax1.axvline(x=0.1, color='black', linestyle='--', alpha=0.4)
ax1.annotate('t=0.1', xy=(0.1, 0.42), fontsize=9)
ax1.set_title('Decision Curve Analysis (RECOMMENDED)', fontweight='bold')

plot_expected_cost_curve(y_true, y_prob, ax=ax2)
ax2.set_title('Expected Cost Curve', fontweight='bold')

plt.tight_layout()
plt.show()

print("The model (blue) has higher net benefit than both reference strategies")
print("across the reasonable range of thresholds (0.05 to 0.40).")
print("This confirms the ADNEX model has clinical utility.")

---
## 7. Logistic Recalibration (Platt Scaling)

Since ADNEX underestimates malignancy risk in this population, we can apply **logistic recalibration**:

$$\text{logit}(P_{recal}) = \alpha + \beta \cdot \text{logit}(P_{orig})$$

This is a **rank-preserving** transformation, so discrimination measures remain unchanged.

In [ ]:
from medpred.metrics.calibration import logistic_recalibration
from medpred.evaluate import evaluate_model

y_prob_recal = logistic_recalibration(y_true, y_prob)

results_orig = evaluate_model(y_true, y_prob, threshold=0.1)
results_recal = evaluate_model(y_true, y_prob_recal, threshold=0.1)

# Compare key measures
print(f"{'Measure':<30} {'Original':>10} {'Recalibrated':>13} {'Change':>8}")
print("=" * 65)

comparisons = [
    ('AUROC', 'discrimination', 'auroc'),
    ('O:E ratio', 'calibration', 'oe_ratio'),
    ('Cal. intercept', 'calibration', 'calibration_intercept'),
    ('Cal. slope', 'calibration', 'calibration_slope'),
    ('Brier score', 'overall', 'brier_score'),
    ('Scaled Brier', 'overall', 'scaled_brier'),
    ('Net benefit', 'clinical_utility', 'net_benefit'),
    ('F1 score', 'classification', 'f1_score'),
    ('Accuracy', 'classification', 'accuracy'),
    ('MCC', 'classification', 'mcc'),
]

for name, domain, key in comparisons:
    v_orig = results_orig[domain][key]
    v_recal = results_recal[domain][key]
    diff = v_recal - v_orig
    arrow = '↑' if diff > 0.001 else ('↓' if diff < -0.001 else '=')
    print(f"  {name:<28} {v_orig:>10.3f} {v_recal:>13.3f} {arrow:>4} {diff:+.3f}")

In [ ]:
print("\nKey observations after recalibration:")
print("  • AUROC unchanged (rank-preserving method)")
print("  • All STRICTLY PROPER measures improved (Brier ↓, Scaled Brier ↑)")
print("  • SEMI-PROPER measures improved or unchanged")
print("  • IMPROPER classification measures WORSENED (F1 ↓, accuracy ↓, MCC ↓)")
print("  • This demonstrates WHY properness matters!")
print("  • Improper measures can mislead you into thinking recalibration hurt the model")

In [ ]:
# Visual comparison of calibration before and after
from medpred.visualization import plot_calibration

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

plot_calibration(y_true, y_prob, ax=ax1)
ax1.set_title('Before Recalibration', fontweight='bold')

plot_calibration(y_true, y_prob_recal, ax=ax2)
ax2.set_title('After Recalibration', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 8. Bootstrap Confidence Intervals

The paper uses the **percentile bootstrap method** with 1000 samples for 95% CIs. Here we use 200 samples for speed in the notebook.

In [ ]:
from medpred.evaluate import evaluate_with_ci, results_to_dataframe

results_ci = evaluate_with_ci(y_true, y_prob, threshold=0.1,
                               n_bootstrap=200, random_state=42)

print("Results with 95% Bootstrap Confidence Intervals")
print("=" * 65)

for domain_name, display in [
    ('discrimination', 'DISCRIMINATION'),
    ('calibration', 'CALIBRATION'),
    ('clinical_utility', 'CLINICAL UTILITY'),
]:
    print(f"\n  --- {display} ---")
    for key, val in results_ci[domain_name].items():
        if isinstance(val, dict) and 'point' in val:
            ci_str = f"({val['lower']:.3f} to {val['upper']:.3f})"
            print(f"    {key:<28} {val['point']:.3f}  95% CI: {ci_str}")

---
## 9. Full Recommended Evaluation — Complete Results Table

The paper's Table 1 summarizes all 32 measures with their properness status, focus assessment, and recommendation. Let's reproduce this.

In [ ]:
# Full results table with annotations
results_full = evaluate_model(y_true, y_prob, threshold=0.1)
df = results_to_dataframe(results_full)

# Style: highlight recommended measures
print("Complete Evaluation Table (reproducing paper Table 1)")
print("=" * 90)
print(df.to_string(index=False))

In [ ]:
# Filter to just the recommended measures
recommended = df[df['recommendation'] == 'Recommended']
print("\n★ RECOMMENDED MEASURES (must-report for clinical validation):")
print("=" * 60)
for _, row in recommended.iterrows():
    print(f"  {row['measure']:<30} = {row['value']:.3f}  [{row['domain']}]")

print("\n  + Calibration plot (see above)")
print("  + Risk distribution plot (see below)")

In [ ]:
from medpred.visualization import plot_risk_distribution

fig, ax = plt.subplots(figsize=(7, 5))
plot_risk_distribution(y_true, y_prob, ax=ax)
ax.set_title('Risk Distribution by Outcome (RECOMMENDED)', fontweight='bold')
plt.show()

In [ ]:
# Generate the complete recommended set of plots in one figure
from medpred.visualization import plot_full_evaluation

fig = plot_full_evaluation(y_true, y_prob, auroc=disc['auroc'])
plt.show()

---
## Summary of Key Findings

### ADNEX Model Performance (External Validation, n=894)

| Aspect | Finding |
|--------|---------|
| **Discrimination** | AUROC = 0.911 — Excellent ability to separate benign from malignant |
| **Calibration** | O:E = 1.23 — Model underestimates risk by ~23% (tertiary care setting) |
| **Clinical Utility** | NB = 0.443 at t=0.1 — Model outperforms treat-all and treat-none strategies |
| **After Recalibration** | Calibration improved to O:E = 1.000; Brier improved; discrimination unchanged |

### Framework Lessons

1. **Always report:** AUROC + calibration plot + decision curve + risk distribution
2. **Avoid:** F1 score (improper + unclear focus), AUPRC, classification accuracy at clinical thresholds
3. **Proper measures** cannot be fooled — they always favor the correct model in expectation
4. **Clinical utility** is the most important domain for deciding whether to use a model in practice
5. **Class imbalance ≠ misclassification costs** — don't conflate epidemiology with decision theory

In [ ]:
print("Notebook complete!")
print("\nReference: Van Calster et al. (2025). Evaluation of performance measures")
print("in predictive AI models to support medical decisions: overview and guidance.")
print("The Lancet Digital Health, 7, 100916.")